In [44]:
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)  # This shows full column content
pd.set_option('display.width', None)  # Auto-detect display width
pd.set_option('display.max_seq_items', None)  # Show all items in lists

In [45]:
df = pd.read_csv('../data/dcInbox/dcinbox_export_114.csv')
unnamed_cols = df.columns.str.contains('^Unnamed')
df = df.loc[:, ~unnamed_cols].copy()
df = df[pd.to_numeric(df['Unix Timestamp'], errors='coerce').notna()].copy()
df['datetime'] = pd.to_datetime(df['Unix Timestamp'], unit='ms')

# Sort chronologically
df = df.sort_values('datetime').reset_index(drop=True)

# Filter data for February and March
# df = df[df['datetime'].between('2016-01-01', '2016-12-31')]
df = df[df['Chamber'] == 'House']
df.info()

/var/folders/02/c1hvrmj11kx0z457p84l6pbc0000gn/T/ipykernel_67272/1599272222.py:1: DtypeWarning: Columns (2,4,10,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139,140,141,142,143,144,145,146,147,148,149,150,151,152,153,154,155,156,157,158,159,160,161,162,163,164,165,166,167,168,169,170,171,172,173,174,175,176,177,178,179,180,181,182,183,184,185,186,187,188,189,190,191,192,193,194,195,196,197,198,199,200,201,202,203,204,205,206,207,208,209,210,211,212,213,214,215,216,217,218,219,220,221,222,223,224,225,226,227,228,229,230,231,232,233,234,235,236,237,238,239,240,241,242,243,244,245,246,247,248,249,250,251,252,253,254,255,256,25

<class 'pandas.core.frame.DataFrame'>
Index: 20582 entries, 3 to 24132
Data columns (total 16 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   Subject         20581 non-null  object        
 1   Body            20582 non-null  object        
 2   Unix Timestamp  20582 non-null  object        
 3   BioGuide ID     20582 non-null  object        
 4   Congress        20582 non-null  object        
 5   First Name      20582 non-null  object        
 6   Last Name       20582 non-null  object        
 7   Date of Birth   20582 non-null  object        
 8   Gender          20582 non-null  object        
 9   State           20582 non-null  object        
 10  District        20582 non-null  object        
 11  Party           20582 non-null  object        
 12  Chamber         20582 non-null  object        
 13  Nickname        1084 non-null   object        
 14  ID              20582 non-null  object        
 15  datetim

/var/folders/02/c1hvrmj11kx0z457p84l6pbc0000gn/T/ipykernel_67272/1599272222.py:5: FutureWarning: The behavior of 'to_datetime' with 'unit' when parsing strings is deprecated. In a future version, strings will be parsed as datetime strings, matching the behavior without a 'unit'. To retain the old behavior, explicitly cast ints or floats to numeric type before calling to_datetime.
  df['datetime'] = pd.to_datetime(df['Unix Timestamp'], unit='ms')


In [46]:
THEMATIC_LEXICON = {
    # Shared/Bipartisan Categories
    'Healthcare': [
        # From both parties
        'health care',
        'healthcare',
        'obamacare',
        'affordable care act',
        'medicare',
        'medicaid',
        'public health',
        'mental health',
        'health',  # Dem-leaning
        'repeal',  # Rep-leaning (context: "repeal Obamacare")
        'repeal and replace',  # Rep-leaning
    ],
    
    'Education': [
        'education',
        'student',  # Dem focus
        'college',  # Dem focus
        'student loan',  # Dem focus
        'community college',  # Dem focus
        'school choice',  # Rep focus
        'parental rights',  # Rep focus
        'curriculum',
        'crt',  # Rep focus
        'critical race theory',  # Rep focus
    ],
    
    'Gun Policy': [
        'gun violence',  # Dem frame
        'gun safety',  # Dem frame
        'gun control',  # Both parties
        'gun reform',  # Dem frame
        'gun',
        'violence',
        'second amendment',  # Rep frame
        '2nd amendment',  # Rep frame
        'gun rights',  # Rep frame
        'right to bear arms',  # Rep frame
    ],
    
    'Immigration': [
        'immigration reform',  # Dem focus
        'immigration policy',
        'border security',  # Both parties
        'border',  # Rep focus
        'southern border',  # Rep focus
        'border crisis',  # Rep focus
        'illegal immigration',  # Rep focus
        'immigration enforcement',  # Rep focus
        'secure the border',  # Rep focus
        'border wall',  # Rep focus
        'sanctuary cities',  # Rep focus
        'illegal',  # Rep focus
        'amnesty',  # Rep focus
    ],
    
    'Social Safety Net & Entitlements': [
        'social security',  # Both parties
        'medicare',  # Note: also in Healthcare - intentional
        'medicaid',  # Note: also in Healthcare - intentional
        'child care',  # Dem focus
        'minimum wage',  # Dem focus
        'wage',  # Dem focus
        'benefit',  # Dem focus
        'assistance',  # Dem focus
        'entitlements',  # Rep frame
        'welfare',  # Rep frame
    ],
    
    'Political Figures - Trump': [
        'trump',
        'donald trump',
        'president trump',
        'make america great',
        'make america great again',
        'maga',
        'america first',
    ],
    
    'Political Figures - Hillary Clinton': [
        'hillary',
        'hillary clinton',
        'clinton',
        'secretary clinton',
        'lock her up',  # Rep-only
    ],
    
    'Political Figures - Obama': [
        'obama',
        'president obama',
        'obama administration',
        'obamas',
        'failed policies',  # Rep frame
    ],
    
    'Congress & Legislation': [
        'legislation',
        'this bill',
        'the bill',
        'bills',
        'house',
        'senate',
        'congress',
        'committee',
        'vote',
        'votes',
        'law',
        'amendment',
        'veto',
        'majority',
        'majority leader',
        'sponsored',
    ],
    
    # Democratic-Leaning Categories
    'Voting Rights': [
        'voting rights',
        'voting rights act',
        'election security',  # Note: also in Rep "Election Integrity"
        'voter',
        'election',
    ],
    
    'Civil Rights': [
        'civil rights',
        "women's rights",
        'equality',
        'equal',
        'civil',
    ],
    
    'Reproductive Rights': [
        'reproductive rights',
        'abortion rights',
        'abortion access',
        'abortion care',
    ],
    
    'Climate & Energy Policy': [
        'climate change',
        'renewable energy',
        'clean energy',
        'green energy',
        'climate',
    ],
    
    'Criminal Justice Reform': [
        'criminal justice',
        'police reform',
        'criminal justice reform',
    ],
    
    'Housing & Community': [
        'housing',
        'affordable',
        'community',
        'neighborhood',
    ],
    
    # Republican-Leaning Categories
    'Election Integrity': [
        'election integrity',
        'voter fraud',
        'voter id',
        'election reform',
        'voter verification',
    ],
    
    'Law Enforcement & Crime': [
        'law enforcement',
        'police',
        'law and order',
        'crime',
        'order',
        'public safety',
    ],
    
    'National Security & Defense': [
        'national security',
        'national defense',
        'homeland security',
        'defense',
        'military',
        'veterans',
        'armed forces',
        'defense spending',
    ],
    
    'Terrorism & Foreign Threats': [
        'terrorism',
        'terrorist',
        'terrorists',
        'isis',
        'terror',
        'guantanamo',
    ],
    
    'Iran & Nuclear Policy': [
        'iran',
        'iran deal',
        'nuclear',
        'sanctions',
        'nuclear deal',
    ],
    
    'Economy & Taxes': [
        'taxes',
        'tax',
        'tax cuts',
        'tax relief',
        'tax reform',
        'tax code',
        'economy',
        'jobs',
        'economic growth',
        'small business',
        'business',
        'regulations',
        'regulatory',
        'regulation',
        'government spending',
        'spending',
        'budget',
        'deficit',
        'dollars',
    ],
    
    'Energy Independence': [
        'energy',
        'energy independence',
        'energy production',
        'oil and gas',
        'fossil fuels',
        'pipeline',
        'epa',
    ],
    
    'Social Issues & Values': [
        'religious freedom',
        'religious liberty',
        'life',
        'pro-life',
        'unborn',
        'sanctity of life',
        'family values',
        'traditional values',
        'planned parenthood',
        'abortion',  # Note: different frame than Dem "abortion rights"
        'god',
        'god bless',
        'bless',
        'christmas',
    ],
    
    'Government Overreach': [
        'big government',
        'government overreach',
        'overreach',
        'bureaucracy',
        'bureaucrats',
        'federal government',
        'government',
        'mandates',
        'federal overreach',
        'states rights',
        'states',
        'freedom',
        'liberty',
        'conservative',
        'constitution',
        'constitutional',
        'swamp',
        'drain the swamp',
    ],
    
    'Supreme Court & Judiciary': [
        'supreme court',
        'judicial',
        'judges',
        'department of justice',
        'courts',
    ],
    
    'Executive Power': [
        'executive',
        'executive order',
        'power',
        'accountability',
        'accountable',
        'oversight',
        'transparency',
    ],
    
    'Patriotic Rhetoric': [
        'american',
        'american people',
        'america first'
    ],

    'Media & Press': [
        # Generic media terms
        'media',
        'press',
        'news',
        'journalist',
        'journalists',
        'reporter',
        'reporters',
        'journalism',
        
        # Specific outlets - mainstream
        'cnn',
        'fox news',
        'msnbc',
        'nbc',
        'cbs',
        'abc news',
        'new york times',
        'washington post',
        'wall street journal',
        'usa today',
        'associated press',
        'reuters',
        
        # Negative/dismissive frames (Rep-leaning)
        'fake news',
        'mainstream media',
        'msm',  # mainstream media acronym
        'liberal media',
        'biased media',
        'media bias',
        'dishonest media',
        'corrupt media',
        'media elite',
        'media elites',
        'left-wing media',
        'lamestream media',
        'enemy of the people',  # Trump phrase about media
        
        # Neutral/positive frames
        'free press',
        'freedom of the press',
        'independent media',
        'investigative journalism',
        
        # Actions related to media
        'fake story',
        'false reporting',
        'media coverage',
        'press coverage',
        'news coverage',
        'reported',
        'reporting',
    ],
    'Trump Name Calling': [        
        # Politicians - Hillary Clinton  
        'crooked hillary',
        'crazy hillary',
        'beautiful hillary',
        
        # Politicians - Trump Critics
        'lyin ted',
        'beautiful ted',  # Later nickname for Cruz
        'little marco',
        'low energy jeb',
        'pocahontas',  # Elizabeth Warren
        'goofy elizabeth warren',
        'crazy bernie',
        'mini mike',  # Bloomberg
        'alfred e neuman',  # Buttigieg
        'sleepy eyes',  # Chuck Todd
        
        # Politicians - Ron DeSantis
        'ron desanctimonious',
        'meatball ron',
        'ron desanctus',
        'tiny d',
        
        # Politicians - Kamala Harris
        'laffin kamala',
        'kamabla',
        'comrade kamala',
        'crazy kamala',
        
        # Politicians - Others
        'liddle bob corker',
        'liddle adam schiff',
        'little adam schitt',
        'pencil neck',  # Adam Schiff
        'crying chuck',  # Schumer
        'crazy nancy',  # Pelosi
        'nancy antoinette',
        'fat jerry',  # Nadler
        'wacky jacky',  # Rosen
        'sloppy steve',  # Bannon
        'shifty schiff',
        'cryin chuck schumer',
        'leakin james comey',
        'slimeball comey',
        'sneaky dianne feinstein',
        'jeff flakey',
        'cheatin obama',
        'da nang dick',  # Blumenthal
        'fat alvin',  # Bragg
        'birdbrain',  # Nikki Haley
        'delusional mike pence',
        'wacky omarosa',
        'governor moonbeam',  # Jerry Brown
        'gavin newscum',
        'old crow',  # McConnell
        'broken old crow',
        'cocaine mitch',
        'low iq war hawk',  # Liz Cheney
        'weirdo tom steyer',
        'tampon tim',  # Tim Walz
        '1 for 38 john',  # Kasich
        
        # Media figures
        'fake news',
        'fake tapper',  # Jake Tapper
        'little george',  # Stephanopoulos  
        'sleepy eyes chuck todd',
        'crazy mika',
        'psycho joe',  # Scarborough
        'morning psycho',
        'don lemon',
        'lyin brian williams',
        'failing new york times',
        'amazon washington post',
        'jeff bozo',  # Bezos
        'little katy',  # Katy Tur
        'liddle peter baker',
        
        # International leaders
        'little rocket man',  # Kim Jong Un
        'rocket man',
        'animal assad',
        'governor justin',  # Trudeau
        
        # Groups/Organizations
        '13 angry democrats',
        'aoc plus 3',
        'deface the nation',
        'democrat party',  # (vs Democratic Party)
        'radical left',
        'do nothing democrats',
    ]
}

In [47]:
def extract_email_level_matches(df, lexicon):
    """
    Extract phrase and theme matches for each email.
    
    Returns a dataframe with email-level details including matched phrases and themes.
    """
    results = []
    
    for idx, row in df.iterrows():
        full_text = row['Body'].lower()
        
        matched_phrases = []
        matched_themes = set()
        
        # Check each theme and its phrases
        for theme, phrases in lexicon.items():
            for phrase in phrases:
                if phrase.lower() in full_text:
                    matched_phrases.append(phrase)
                    matched_themes.add(theme)
        
        # Only include emails that matched at least one phrase
        if matched_phrases:
            results.append({
                'first_name': row.get('First Name', ''),
                'last_name': row.get('Last Name', ''),
                'party': row.get('Party', ''),
                'email_id': row.get('ID'),  # Use index if no 'id' column
                'subject': row.get('Subject', ''),
                'matched_phrases': matched_phrases,
                'themes': list(matched_themes),
                'email': row.get('Body', '')
            })
    
    return pd.DataFrame(results)

def process_lexicon(dataframe, lexicon):
    """Process thematic lexicon and return theme and phrase results"""
    
    # Combine subject and body
    df_copy = dataframe.copy()
    df_copy['full_text'] = (df_copy['Subject'].fillna('').astype(str) + ' ' + 
                            df_copy['Body'].fillna('').astype(str)).str.lower()
    
    phrase_results = []
    theme_results = []
    
    for theme, phrases in lexicon.items():
        # Track which emails mention this theme (any phrase)
        theme_mask = pd.Series([False] * len(df_copy), index=df_copy.index)
        
        for phrase in phrases:
            # Count emails containing this specific phrase
            phrase_mask = df_copy['full_text'].str.contains(phrase, case=False, regex=False)
            count = phrase_mask.sum()
            percentage = (count / len(df_copy)) * 100
            
            phrase_results.append({
                'theme': theme,
                'phrase': phrase,
                'email_count': count,
                'percentage': percentage
            })
            
            # Add to theme mask
            theme_mask = theme_mask | phrase_mask
        
        # Calculate theme-level coverage
        theme_count = theme_mask.sum()
        theme_percentage = (theme_count / len(df_copy)) * 100
        
        theme_results.append({
            'theme': theme,
            'email_count': theme_count,
            'percentage': theme_percentage,
            'num_phrases': len(phrases)
        })
    
    return theme_results, phrase_results

In [48]:
dem_df = df[df['Party'] == 'Democrat'].copy()
print(f"Democrat emails: {len(dem_df)}")

theme_results, phrase_results = process_lexicon(dem_df, THEMATIC_LEXICON)


# Create results dataframes
phrase_df = pd.DataFrame(phrase_results).sort_values('percentage', ascending=False)
theme_df = pd.DataFrame(theme_results).sort_values('percentage', ascending=False)

# Display theme-level summary
print("\n" + "="*70)
print("THEME COVERAGE (emails mentioning ANY phrase in theme)")
print("="*70)

for i, row in enumerate(theme_df.itertuples(), 1):
    print(f"{i:2d}. {row.theme:25s}: {row.email_count:5d} emails ({row.percentage:5.2f}%) [{row.num_phrases} phrases]")

# Display top individual phrases
print("\n" + "="*70)
print("TOP INDIVIDUAL PHRASES BY COVERAGE")
print("="*70)

for i, row in enumerate(phrase_df.head(20).itertuples(), 1):
    print(f"{i:2d}. {row.phrase:30s} ({row.theme:20s}): {row.email_count:5d} ({row.percentage:5.2f}%)")

Democrat emails: 5836

THEME COVERAGE (emails mentioning ANY phrase in theme)
 1. Congress & Legislation   :  5619 emails (96.28%) [16 phrases]
 2. Media & Press            :  4736 emails (81.15%) [44 phrases]
 3. Economy & Taxes          :  3480 emails (59.63%) [19 phrases]
 4. Government Overreach     :  3232 emails (55.38%) [18 phrases]
 5. Housing & Community      :  3129 emails (53.62%) [4 phrases]
 6. Energy Independence      :  2865 emails (49.09%) [7 phrases]
 7. Patriotic Rhetoric       :  2851 emails (48.85%) [3 phrases]
 8. Social Safety Net & Entitlements:  2788 emails (47.77%) [10 phrases]
 9. Healthcare               :  2774 emails (47.53%) [11 phrases]
10. Education                :  2593 emails (44.43%) [10 phrases]
11. National Security & Defense:  2482 emails (42.53%) [8 phrases]
12. Social Issues & Values   :  1744 emails (29.88%) [14 phrases]
13. Law Enforcement & Crime  :  1723 emails (29.52%) [6 phrases]
14. Executive Power          :  1676 emails (28.72%) [7 phra

In [49]:
rep_df = df[df['Party'] == 'Republican'].copy()
print(f"Republican emails: {len(rep_df)}")

# Combine subject and body
print("\nCombining subject and body text...")
rep_df['full_text'] = (rep_df['Subject'].fillna('').astype(str) + ' ' + 
                        rep_df['Body'].fillna('').astype(str)).str.lower()

theme_results, phrase_results = process_lexicon(rep_df, THEMATIC_LEXICON)


# Create results dataframes
phrase_df = pd.DataFrame(phrase_results).sort_values('percentage', ascending=False)
theme_df = pd.DataFrame(theme_results).sort_values('percentage', ascending=False)

# Display theme-level summary
print("\n" + "="*70)
print("THEME COVERAGE (emails mentioning ANY phrase in theme)")
print("="*70)

for i, row in enumerate(theme_df.itertuples(), 1):
    print(f"{i:2d}. {row.theme:35s}: {row.email_count:5d} emails ({row.percentage:5.2f}%) [{row.num_phrases} phrases]")

# Display top individual phrases
print("\n" + "="*70)
print("TOP INDIVIDUAL PHRASES BY COVERAGE")
print("="*70)

for i, row in enumerate(phrase_df.head(30).itertuples(), 1):
    print(f"{i:2d}. {row.phrase:30s} ({row.theme:30s}): {row.email_count:5d} ({row.percentage:5.2f}%)")

Republican emails: 14746

Combining subject and body text...

THEME COVERAGE (emails mentioning ANY phrase in theme)
 1. Congress & Legislation             : 14551 emails (98.68%) [16 phrases]
 2. Media & Press                      : 12381 emails (83.96%) [44 phrases]
 3. Government Overreach               : 10826 emails (73.42%) [18 phrases]
 4. Economy & Taxes                    : 10325 emails (70.02%) [19 phrases]
 5. Patriotic Rhetoric                 :  8684 emails (58.89%) [3 phrases]
 6. National Security & Defense        :  8446 emails (57.28%) [8 phrases]
 7. Energy Independence                :  8034 emails (54.48%) [7 phrases]
 8. Executive Power                    :  7106 emails (48.19%) [7 phrases]
 9. Healthcare                         :  6638 emails (45.02%) [11 phrases]
10. Social Issues & Values             :  6018 emails (40.81%) [14 phrases]
11. Housing & Community                :  5705 emails (38.69%) [4 phrases]
12. Political Figures - Obama          :  5644 email

In [50]:
emails_email_matches_df = extract_email_level_matches(df, THEMATIC_LEXICON)

# Display summary statistics
print(f"\nTotal emails with theme matches: {len(emails_email_matches_df)}")
print(f"Percentage of emails with matches: {len(emails_email_matches_df)/len(rep_df)*100:.2f}%")

# Show a few examples
print("\nSample of email-level matches:")

emails_email_matches_df.head(5)
emails_email_matches_df.to_csv('../data/dcInbox/email_themes_dcinbox_114.csv', index=False)


Total emails with theme matches: 20487
Percentage of emails with matches: 138.93%

Sample of email-level matches:


In [51]:
emails_email_matches_df.head(5)

first_name last_name       party email_id  \
0      Jason     Smith  Republican   101001   
1      Henry   Cuellar    Democrat   101000   
2     Thomas    Massie  Republican   100999   
3    Richard     Hanna  Republican   100997   
4      Louie   Gohmert  Republican   100998   

                                                                                                   subject  \
0                                               Congressman Jason Smith Capitol Report: The Year in Review   
1                                                                                     Congressional Report   
2  PRESS RELEASE: U.S. Representative Thomas Massie Announces He Will Not Vote to Re-elect Speaker Boehner   
3                                                                                        News from Upstate   
4                                               Rep. Louie Gohmert Announces Run for Speaker of the House    

                                                                                                                                                                                                                                                       matched_phrases  \
0  [healthcare, health, second amendment, assistance, obama, president obama, legislation, bills, house, senate, congress, amendment, veterans, tax, economy, jobs, regulations, regulation, spending, budget, pipeline, epa, life, overreach, states, freedom, media]   
1                                                      [healthcare, health, education, student, benefit, house, congress, law, criminal justice, housing, community, neighborhood, veterans, economy, epa, life, christmas, government, states, american, press, news]   
2                                                                                                                                              [bills, house, congress, committee, vote, order, isis, spending, christmas, constitution, constitutional, media, press]   
3                                                                                                                                                                                     [student, house, congress, community, order, business, epa, supreme court, news]   
4                                                                                                                           [house, senate, congress, committee, vote, votes, majority, voter, election, crime, homeland security, terrorism, terror, power, american]   

                                                                                                                                                                                                                                                                                themes  \
0                                        [Media & Press, Gun Policy, Social Safety Net & Entitlements, Congress & Legislation, Energy Independence, Economy & Taxes, National Security & Defense, Political Figures - Obama, Social Issues & Values, Government Overreach, Healthcare]   
1  [Criminal Justice Reform, Media & Press, Congress & Legislation, Social Safety Net & Entitlements, Education, Energy Independence, Economy & Taxes, Patriotic Rhetoric, National Security & Defense, Social Issues & Values, Government Overreach, Healthcare, Housing & Community]   
2                                                                                                                         [Media & Press, Congress & Legislation, Law Enforcement & Crime, Social Issues & Values, Government Overreach, Economy & Taxes, Terrorism & Foreign Threats]   
3                                                                                                                    [Media & Press, Congress & Legislation, Education, Energy Independence, Law Enforcement & Crime, Supreme Court & Judiciary, Economy & Taxes, Housing & Community]   
4                                                  

In [52]:

# Filter for rows where 'Trump Name Calling' is in the themes list
trump_namecalling_df = emails_email_matches_df[emails_email_matches_df['themes'].apply(lambda x: 'Trump Name Calling' in x)]

# Display the results
print(f"Total emails with Trump Name Calling theme: {len(trump_namecalling_df)}")
print(f"Percentage: {len(trump_namecalling_df)/len(df)*100:.2f}%")

# View the dataframe
trump_namecalling_df.head(5)


Total emails with Trump Name Calling theme: 35
Percentage: 0.17%


first_name last_name       party email_id  \
3464      Steve   Scalise  Republican    97005   
4119       Doug   LaMalfa  Republican    96231   
6804       Paul     Gosar  Republican   116904   
7048       Evan   Jenkins  Republican   116626   
7057       Bill   Johnson  Republican   116613   

                                            subject  \
3464     Meeting the Needs of Military and Veterans   
4119             A Message from Congressman LaMalfa   
6804              This is a HUGE blow to the EPA...   
7048  Rep. Evan Jenkins: Welcome To My E-Newsletter   
7057      Update from Eastern and Southeastern Ohio   

                                                                                                                                                                                                                                                                                                                                                                                                    matched_phrases  \
3464  [health care, obamacare, medicare, health, repeal, medicare, obama, president obama, house, senate, congress, law, amendment, majority, national defense, defense, military, veterans, tax, economy, jobs, budget, epa, religious freedom, life, pro-life, abortion, god, god bless, bless, government, freedom, conservative, constitution, constitutional, oversight, american, media, press, radical left]   
4119                                                                                                                                                                                                                                                                   [benefit, assistance, this bill, house, congress, housing, veterans, tax, government spending, spending, epa, government, freedom, old crow]   
6804                                                       [obama, president obama, legislation, bills, house, senate, congress, vote, law, amendment, civil, tax, economy, jobs, business, regulations, regulation, dollars, epa, big government, overreach, bureaucrats, government, states, constitution, constitutional, supreme court, judicial, executive, power, american, media, press, news, radical left]   
7048                                                                                                                                                                                                                                                                                            [benefit, legislation, house, congress, national security, isis, iran, iran deal, nuclear, media, news, pocahontas]   
7057                                                                                                                                           [healthcare, health, education, student, assistance, obama, president obama, obama administration, house, congress, housing, community, iran, iran deal, jobs, business, regulations, regulation, energy, oil and gas, epa, power, media, press, news, radical left]   

                                                                                                                                                                                                                                                                                          themes  \
3464  [Media & Press, Congress & Legislation, Social Safety Net & Entitlements, Trump Name Calling, Energy Independence, Economy & Taxes, Patriotic Rhetoric, National Security & Defense, Political Figures - Obama, Social Issues & Values, Government Overreach, Healthcare, Executive Power]   
4119                                                                                                [Congress & Legislation, Social Safety Net & Entitlements, Trump Name Calling, Energy Independence, National Security & Defense, Government Overreach, Economy & Taxes, Housing & Community]   
6804                                                  